# 🔥 Gas Turbine Combustion Kinetics & Stiff Neural ODEs
### Modeling Chemical Stiffness, Autoignition Delay, and Thermal NOx Formation
**Author:** Mohammed Basra (`mohammedbasra-star`)  

This tutorial demonstrates:
1. **Stiff Chemical Kinetics:** Multi-step methane oxidation with Arrhenius kinetics & Zeldovich $NO_x$.
2. **Implicit Radau IIA Integration:** Ground-truth stiff trajectory simulation and Jacobian spectral analysis.
3. **Stiff Continuous Neural ODE:** Modeling multi-timescale reaction runaway and enforcing physical element conservation.

In [ ]:
import os, sys
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add repository root
sys.path.append(os.path.abspath('..'))

from chemistry.stiff_solver import solve_autoignition_trajectory
from chemistry.thermo_data import SPECIES_NAMES
from models.stiff_neural_ode import StiffNeuralODE

## 1. Simulating Gas Turbine Autoignition with Radau IIA Stiff Integrator

In [ ]:
sol = solve_autoignition_trajectory(T0=1250.0, phi=1.0, pressure_bar=15.0, t_end=0.05, num_time_steps=150)
time_ms = sol['t'] * 1000.0

print(f"Ignition Delay (tau_ign): {sol['t_ignition']*1000.0:.2f} ms")
print(f"Adiabatic Flame Temperature: {sol['T_flame']:.1f} K")
print(f"Thermal NOx Concentration: {sol['NO_ppm']:.2f} ppm")
print(f"Chemical Stiffness Ratio: {sol['stiffness_ratio']:.2e}")

plt.figure(figsize=(8, 4))
plt.plot(time_ms, sol['y'][0], 'k-', lw=2.5)
plt.axvline(sol['t_ignition']*1000.0, color='r', linestyle='--', label=f'Ignition Delay = {sol["t_ignition"]*1000.0:.2f} ms')
plt.xlabel('Time (ms)')
plt.ylabel('Combustor Temperature T (K)')
plt.title('Gas Turbine Combustor Autoignition Transient')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 2. Chemical Species Transients and Radical Depletion

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(time_ms, sol['y'][1], label='CH4 (Fuel)', lw=2)
plt.plot(time_ms, sol['y'][2], label='O2 (Oxidizer)', lw=2)
plt.plot(time_ms, sol['y'][3], label='CO2 (Product)', lw=2)
plt.plot(time_ms, sol['y'][4], label='H2O (Product)', lw=2)
plt.plot(time_ms, sol['y'][5], label='CO (Intermediate)', lw=2)
plt.xlabel('Time (ms)')
plt.ylabel('Mass Fraction (Y_k)')
plt.title('Species Mass Fraction Dynamics')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 3. Continuous Stiff Neural ODE Inference

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StiffNeuralODE(state_dim=9, hidden_dim=128).to(device)

checkpoint_path = '../results/best_combustion_node.pt'
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    print('Loaded pretrained Stiff Neural ODE checkpoint!')
else:
    print('Model initialized (run train.py to train complete weights)')

time_grid = torch.from_numpy(sol['t']).float().to(device)
z0 = torch.from_numpy(sol['y'][:, 0]).float().unsqueeze(0).to(device)
z0[:, 0] /= 2500.0  # normalize T

with torch.no_grad():
    pred_traj = model(z0, time_grid).squeeze(0).cpu().numpy()

T_pred = pred_traj[0] * 2500.0

plt.figure(figsize=(8, 4))
plt.plot(time_ms, sol['y'][0], 'k-', lw=2.5, label='Exact Radau5')
plt.plot(time_ms, T_pred, 'r--', lw=2, label='Stiff Neural ODE')
plt.xlabel('Time (ms)')
plt.ylabel('Temperature (K)')
plt.title('Neural ODE Continuous State-Space Tracking')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()